# Assignment 5 — Catching Data Leakage Before It Catches You
### Feature Engineering & MLOps · Data Leakage: Target Leakage & Preprocessing Leakage

**Objective:** demonstrate, with real numbers, both **target leakage** (a feature that is secretly a *result* of the label rather than a predictor of it) and **preprocessing leakage** (fitting a transformation on data the model should never have seen yet) — then correctly fix both.

**Dataset:** `customer_churn_a5.csv` — 600 telecom customers.

| Column | Description |
|---|---|
| `tenure_months` | Months this customer has held their account (legitimate) |
| `monthly_charges` | Current monthly bill amount (legitimate) |
| `contract_type` | Month-to-month / One year / Two year (legitimate) |
| `support_calls` | Number of support calls placed so far (legitimate) |
| `days_since_cancellation` | Days since the account was cancelled — **only present if churn=1** |
| `final_bill_amount` | Closing balance at account closure — **only present if churn=1** |
| `churn` | Target — did this customer cancel their service? |

## Setup — Imports

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

RANDOM_STATE = 42
print("Libraries loaded.")

Libraries loaded.


## Load & Inspect the Dataset

In [2]:
df = pd.read_csv('customer_churn_a5.csv')
print("Shape:", df.shape)
df.head(10)

Shape: (600, 8)


,customer_id,tenure_months,monthly_charges,contract_type,support_calls,days_since_cancellation,final_bill_amount,churn
0,50001,29,56.48,Month-to-month,2,70.0,112.43,1
1,50002,30,101.00,Month-to-month,4,90.0,169.95,1
2,50003,30,89.05,Two year,1,NaN,NaN,0
3,50004,57,73.25,One year,2,NaN,NaN,0
4,50005,50,82.26,One year,1,NaN,NaN,0
5,50006,5,87.45,One year,2,92.0,169.27,1
6,50007,28,63.09,Month-to-month,1,NaN,NaN,0
7,50008,1,68.34,One year,2,NaN,NaN,0
8,50009,59,91.41,Month-to-month,2,56.0,114.41,1
9,50010,16,92.57,Month-to-month,3,78.0,157.89,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              600 non-null    int64  
 1   tenure_months            600 non-null    int64  
 2   monthly_charges          600 non-null    float64
 3   contract_type            600 non-null    str    
 4   support_calls            600 non-null    int64  
 5   days_since_cancellation  149 non-null    float64
 6   final_bill_amount        149 non-null    float64
 7   churn                    600 non-null    int64  
dtypes: float64(3), int64(4), str(1)
memory usage: 37.6 KB


In [4]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nChurn distribution:")
print(df['churn'].value_counts())

Missing values per column:
customer_id                  0
tenure_months                0
monthly_charges              0
contract_type                0
support_calls                0
days_since_cancellation    451
final_bill_amount          451
churn                        0
dtype: int64

Churn distribution:
churn
0    451
1    149
Name: count, dtype: int64


**Observation:** the dataset has 600 rows and 8 columns. `days_since_cancellation` and `final_bill_amount` each have 451 missing values — exactly matching the number of `churn=0` customers, which is our first clue that these two columns only exist *after* a customer has already churned.

---
## Section 4.1 — Target Leakage: Prove It, Don't Just Assert It

### Step 1 — Confirm `days_since_cancellation` is only populated for churned customers
We prove this with a single line of code rather than eyeballing it.

In [5]:
non_null_cancellation_count = df['days_since_cancellation'].notna().sum()
churn_count = (df['churn'] == 1).sum()

print(f"Rows with non-null days_since_cancellation: {non_null_cancellation_count}")
print(f"Rows with churn == 1                       : {churn_count}")

# Single line proof that the two counts are exactly equal
proof = df['days_since_cancellation'].notna().sum() == (df['churn'] == 1).sum()
print(f"\nDoes non-null count exactly equal churn=1 count? {proof}")

Rows with non-null days_since_cancellation: 149
Rows with churn == 1                       : 149

Does non-null count exactly equal churn=1 count? True


**Confirmed:** every single row with a non-null `days_since_cancellation` is a churned customer, and there are no exceptions in either direction. This column literally cannot exist for a customer at the moment we'd need to predict their churn — it is generated *by* the churn event itself.

### Step 2 — Correlation Check: `final_bill_amount` vs. a Legitimate Feature

In [6]:
df_filled = df.copy()
df_filled['final_bill_amount'] = df_filled['final_bill_amount'].fillna(-1)
df_filled['days_since_cancellation'] = df_filled['days_since_cancellation'].fillna(-1)

corr_leaked = df_filled['final_bill_amount'].corr(df_filled['churn'])
corr_legit = df_filled['tenure_months'].corr(df_filled['churn'])

print(f"Correlation of final_bill_amount (missing filled with -1) with churn: {corr_leaked:.4f}")
print(f"Correlation of tenure_months (a genuinely legitimate feature) with churn: {corr_legit:.4f}")
print(f"\nRatio of magnitudes: {abs(corr_leaked) / abs(corr_legit):.1f}x stronger")

Correlation of final_bill_amount (missing filled with -1) with churn: 0.9421
Correlation of tenure_months (a genuinely legitimate feature) with churn: -0.3194

Ratio of magnitudes: 2.9x stronger


**Observation:** `final_bill_amount`'s correlation with `churn` is dramatically higher in magnitude than a genuinely predictive, legitimate feature like `tenure_months`. This is a red flag on its own — real-world customer behavior is rarely predicted this cleanly by a single field, and the reason is that `final_bill_amount` isn't predicting churn at all; its very existence (a non-(-1) value) IS churn, restated as a number.

### Step 3 — Model A: Legitimate Features Only

In [7]:
legit_features = ['tenure_months', 'monthly_charges', 'contract_type', 'support_calls']

X_a = pd.get_dummies(df[legit_features], columns=['contract_type'], drop_first=True)
y = df['churn']

X_a_train, X_a_test, y_train, y_test = train_test_split(
    X_a, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

scaler_a = StandardScaler()
X_a_train_scaled = scaler_a.fit_transform(X_a_train)
X_a_test_scaled = scaler_a.transform(X_a_test)

model_a = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_a.fit(X_a_train_scaled, y_train)

pred_a = model_a.predict(X_a_test_scaled)
proba_a = model_a.predict_proba(X_a_test_scaled)[:, 1]

acc_a = accuracy_score(y_test, pred_a)
auc_a = roc_auc_score(y_test, proba_a)

print("MODEL A — legitimate features only")
print(f"  Test Accuracy: {acc_a:.4f}")
print(f"  Test ROC-AUC : {auc_a:.4f}")

MODEL A — legitimate features only
  Test Accuracy: 0.8667
  Test ROC-AUC : 0.9388


### Step 4 — Model B: Legitimate Features + Leaked Features

In [8]:
leaked_features = ['days_since_cancellation', 'final_bill_amount']

df_b = df.copy()
df_b['days_since_cancellation'] = df_b['days_since_cancellation'].fillna(-1)
df_b['final_bill_amount'] = df_b['final_bill_amount'].fillna(-1)

X_b = pd.get_dummies(df_b[legit_features + leaked_features], columns=['contract_type'], drop_first=True)

# IMPORTANT: use the SAME train/test split indices as Model A for a fair, apples-to-apples comparison
X_b_train, X_b_test = X_b.loc[X_a_train.index], X_b.loc[X_a_test.index]

scaler_b = StandardScaler()
X_b_train_scaled = scaler_b.fit_transform(X_b_train)
X_b_test_scaled = scaler_b.transform(X_b_test)

model_b = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_b.fit(X_b_train_scaled, y_train)

pred_b = model_b.predict(X_b_test_scaled)
proba_b = model_b.predict_proba(X_b_test_scaled)[:, 1]

acc_b = accuracy_score(y_test, pred_b)
auc_b = roc_auc_score(y_test, proba_b)

print("MODEL B — legitimate features + LEAKED features (days_since_cancellation, final_bill_amount)")
print(f"  Test Accuracy: {acc_b:.4f}")
print(f"  Test ROC-AUC : {auc_b:.4f}")

MODEL B — legitimate features + LEAKED features (days_since_cancellation, final_bill_amount)
  Test Accuracy: 1.0000
  Test ROC-AUC : 1.0000


### Step 5 — The Gap, and What Happens in Production

In [9]:
print(f"Model A (honest)  -> Accuracy: {acc_a:.4f}, ROC-AUC: {auc_a:.4f}")
print(f"Model B (leaked)  -> Accuracy: {acc_b:.4f}, ROC-AUC: {auc_b:.4f}")
print(f"\nGap  -> Accuracy: {acc_b - acc_a:+.4f}, ROC-AUC: {auc_b - auc_a:+.4f}")

Model A (honest)  -> Accuracy: 0.8667, ROC-AUC: 0.9388
Model B (leaked)  -> Accuracy: 1.0000, ROC-AUC: 1.0000

Gap  -> Accuracy: +0.1333, ROC-AUC: +0.0612


In [10]:
from IPython.display import Markdown

gap_summary = f"""
**Gap between Model A and Model B:** Model B's ROC-AUC is **{auc_b - auc_a:+.4f}** higher and its accuracy is
**{acc_b - acc_a:+.4f}** higher than Model A's, purely because it was handed two columns
(`days_since_cancellation`, `final_bill_amount`) that are only ever populated **after** a customer has already
churned.

**What would happen in production:** if Model B were deployed to score brand-new applicants — people who
haven't decided anything yet and obviously have no `days_since_cancellation` or `final_bill_amount` value at
all — both of those columns would have to be filled with the same placeholder (-1) for every single applicant,
since a customer who hasn't churned cannot have a cancellation date or closing balance. The model would then
be making its "confident" high-AUC predictions using two constant, information-free inputs, effectively
collapsing back to whatever it learned from the legitimate features alone (or worse, since it was trained to lean
heavily on the now-missing signal). The inflated {auc_b:.4f} ROC-AUC seen in testing would evaporate on day
one in production, and the team would be left wondering why a "great" model suddenly performs like a coin
flip on new applicants.
"""
display(Markdown(gap_summary))


**Gap between Model A and Model B:** Model B's ROC-AUC is **+0.0612** higher and its accuracy is
**+0.1333** higher than Model A's, purely because it was handed two columns
(`days_since_cancellation`, `final_bill_amount`) that are only ever populated **after** a customer has already
churned.

**What would happen in production:** if Model B were deployed to score brand-new applicants — people who
haven't decided anything yet and obviously have no `days_since_cancellation` or `final_bill_amount` value at
all — both of those columns would have to be filled with the same placeholder (-1) for every single applicant,
since a customer who hasn't churned cannot have a cancellation date or closing balance. The model would then
be making its "confident" high-AUC predictions using two constant, information-free inputs, effectively
collapsing back to whatever it learned from the legitimate features alone (or worse, since it was trained to lean
heavily on the now-missing signal). The inflated 1.0000 ROC-AUC seen in testing would evaporate on day
one in production, and the team would be left wondering why a "great" model suddenly performs like a coin
flip on new applicants.


---
## Section 4.2 — Preprocessing Leakage: The Wrong Order vs. The Right Order

### Step 1 — WRONG: Fit `StandardScaler` on the Entire Dataset, Then Split

In [11]:
numeric_legit_features = ['tenure_months', 'monthly_charges', 'support_calls']

X_numeric = df[numeric_legit_features]

# WRONG ORDER: scaler sees the full dataset (including future test rows) before any split happens
scaler_wrong = StandardScaler()
scaler_wrong.fit(X_numeric)   # <-- fit on ALL 600 rows, test rows included

print("WRONG scaler (fit on full dataset BEFORE split):")
print(f"  mean_ : {scaler_wrong.mean_}")
print(f"  scale_: {scaler_wrong.scale_}")

# split only happens afterward, for illustration of what training would then use
X_wrong_train, X_wrong_test = train_test_split(
    X_numeric, test_size=0.25, random_state=RANDOM_STATE)

WRONG scaler (fit on full dataset BEFORE split):
  mean_ : [35.47333333 71.52445     1.64166667]
  scale_: [20.86342786 18.41823555  1.27014326]


### Step 2 — CORRECT: Split First, Then Fit a New `StandardScaler` on the Training Split Only

In [12]:
X_correct_train, X_correct_test = train_test_split(
    X_numeric, test_size=0.25, random_state=RANDOM_STATE)

scaler_correct = StandardScaler()
scaler_correct.fit(X_correct_train)   # <-- fit ONLY on the training rows

print("CORRECT scaler (fit on TRAIN split only, AFTER split):")
print(f"  mean_ : {scaler_correct.mean_}")
print(f"  scale_: {scaler_correct.scale_}")

CORRECT scaler (fit on TRAIN split only, AFTER split):
  mean_ : [35.         71.32582222  1.69777778]
  scale_: [21.06096969 18.71082892  1.27966817]


### Step 3 — Numeric Difference Between the Two Scalers

In [13]:
mean_diff = scaler_wrong.mean_ - scaler_correct.mean_
scale_diff = scaler_wrong.scale_ - scaler_correct.scale_

diff_df = pd.DataFrame({
    'feature': numeric_legit_features,
    'wrong_mean': scaler_wrong.mean_,
    'correct_mean': scaler_correct.mean_,
    'mean_difference': mean_diff,
    'wrong_scale': scaler_wrong.scale_,
    'correct_scale': scaler_correct.scale_,
    'scale_difference': scale_diff
})
diff_df

,feature,wrong_mean,correct_mean,mean_difference,wrong_scale,correct_scale,scale_difference
0,tenure_months,35.473333,35.000000,0.473333,20.863428,21.060970,-0.197542
1,monthly_charges,71.524450,71.325822,0.198628,18.418236,18.710829,-0.292593
2,support_calls,1.641667,1.697778,-0.056111,1.270143,1.279668,-0.009525


**Why this principle matters regardless of how large the gap looks here:** on this particular 600-row dataset, the mean/scale difference between the two scalers may look small — the training split (75% of the data) and the full dataset are similar in distribution simply because of sample size and random chance. But the *mechanism* of the leak doesn't depend on the gap being large: fitting the scaler on the full dataset means the transformation applied to every training row is influenced, even if only slightly, by statistics computed from rows the model is never supposed to see during training (the test rows). At evaluation time this makes the test score an *optimistic estimate* of real-world performance, because the "unseen" test data quietly contributed to how the training data was scaled. On a smaller dataset, a dataset with more outliers, or a dataset where the train/test split isn't as representative, this same mechanism can produce a much larger, more damaging gap — the risk is baked into the *order of operations*, not the specific numbers we happen to see in any one run.

### Step 4 — Rebuild Correctly with a Single `sklearn.Pipeline`
An imputer + scaler + model in **one Pipeline object**, fit only on the training split, should reproduce the exact same scaler statistics as the manual "correct" version above.

In [14]:
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),   # no-op here since these columns have no missing values, but included for a robust, general pipeline
    ('scaler', StandardScaler())
])

# Fit the pipeline ONLY on the training split
numeric_pipeline.fit(X_correct_train)

pipeline_scaler = numeric_pipeline.named_steps['scaler']

print("Pipeline-fitted scaler (fit on TRAIN split only, via Pipeline):")
print(f"  mean_ : {pipeline_scaler.mean_}")
print(f"  scale_: {pipeline_scaler.scale_}")

matches = (np.allclose(pipeline_scaler.mean_, scaler_correct.mean_) and
           np.allclose(pipeline_scaler.scale_, scaler_correct.scale_))
print(f"\nDoes the Pipeline scaler exactly match the manual 'correct' scaler? {matches}")

Pipeline-fitted scaler (fit on TRAIN split only, via Pipeline):
  mean_ : [35.         71.32582222  1.69777778]
  scale_: [21.06096969 18.71082892  1.27966817]

Does the Pipeline scaler exactly match the manual 'correct' scaler? True


---
## Section 4.3 — The Fix: Final Honest, Deployable Model

Using only the legitimate features, with a properly ordered `Pipeline` (split first, fit preprocessing on train only), retrain a final model and confirm it matches Task 4.1's Model A.

In [15]:
# Rebuild legitimate features with one-hot encoding for contract_type (done ONCE, before split,
# since one-hot encoding categories is not a leak-prone "learned statistic" the way scaling/imputing is -
# but to be fully rigorous we still split before fitting any *learned* transformation)
X_final = pd.get_dummies(df[legit_features], columns=['contract_type'], drop_first=True)
y_final = df['churn']

X_final_train, X_final_test, y_final_train, y_final_test = train_test_split(
    X_final, y_final, test_size=0.25, random_state=RANDOM_STATE, stratify=y_final)

final_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

# Fitting the Pipeline on X_final_train fits the scaler AND the model using ONLY training data -
# no leakage of any kind, target or preprocessing
final_pipeline.fit(X_final_train, y_final_train)

final_pred = final_pipeline.predict(X_final_test)
final_proba = final_pipeline.predict_proba(X_final_test)[:, 1]

final_acc = accuracy_score(y_final_test, final_pred)
final_auc = roc_auc_score(y_final_test, final_proba)

print("FINAL FIXED MODEL — legitimate features, properly ordered Pipeline")
print(f"  Test Accuracy: {final_acc:.4f}")
print(f"  Test ROC-AUC : {final_auc:.4f}")

print(f"\nModel A (Section 4.1) -> Accuracy: {acc_a:.4f}, ROC-AUC: {auc_a:.4f}")
print(f"Final Fixed Model      -> Accuracy: {final_acc:.4f}, ROC-AUC: {final_auc:.4f}")

matches_model_a = np.isclose(final_acc, acc_a) and np.isclose(final_auc, auc_a)
print(f"\nDoes the final fixed model match Task 4.1's Model A? {matches_model_a}")

FINAL FIXED MODEL — legitimate features, properly ordered Pipeline
  Test Accuracy: 0.8667
  Test ROC-AUC : 0.9388

Model A (Section 4.1) -> Accuracy: 0.8667, ROC-AUC: 0.9388
Final Fixed Model      -> Accuracy: 0.8667, ROC-AUC: 0.9388

Does the final fixed model match Task 4.1's Model A? True


**Confirmed:** because the final pipeline uses the exact same legitimate features, the exact same train/test split (`random_state=42`, same `test_size`), and fits its scaler only on the training data — exactly like Model A did — the results match. This is the honest, deployable version of the model: no target leakage (no columns that are secretly a result of churn), and no preprocessing leakage (nothing was fit before the split).

---
## Bonus (Optional, +10%) — Can Cross-Validation Leak Too?

We compare 5-fold cross-validation scores when the `StandardScaler` is fit on the whole dataset **before** CV splitting vs. properly wrapping the scaler **inside** the `Pipeline` passed to `cross_val_score` (so it's refit on each fold's training data only).

In [16]:
# WRONG: scaler fit on the FULL dataset before cross-validation ever splits it
scaler_leak = StandardScaler()
X_numeric_leaked_scaled = scaler_leak.fit_transform(X_numeric)   # sees ALL rows, across all folds

model_for_cv = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

cv_scores_leaked = cross_val_score(
    model_for_cv, X_numeric_leaked_scaled, y, cv=5, scoring='roc_auc')

print("Cross-val ROC-AUC scores (scaler fit on FULL dataset before CV - LEAKY):")
print(cv_scores_leaked.round(4))
print(f"Mean: {cv_scores_leaked.mean():.4f}  Std: {cv_scores_leaked.std():.4f}")

Cross-val ROC-AUC scores (scaler fit on FULL dataset before CV - LEAKY):
[0.8015 0.8474 0.7167 0.8093 0.8624]
Mean: 0.8075  Std: 0.0508


In [17]:
# CORRECT: scaler is INSIDE the Pipeline passed to cross_val_score, so it's refit fresh
# on each fold's training portion only, and applied to that fold's held-out portion afterward
cv_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

cv_scores_correct = cross_val_score(
    cv_pipeline, X_numeric, y, cv=5, scoring='roc_auc')

print("Cross-val ROC-AUC scores (scaler INSIDE Pipeline, refit per fold - CORRECT):")
print(cv_scores_correct.round(4))
print(f"Mean: {cv_scores_correct.mean():.4f}  Std: {cv_scores_correct.std():.4f}")

Cross-val ROC-AUC scores (scaler INSIDE Pipeline, refit per fold - CORRECT):
[0.8007 0.8474 0.7167 0.8096 0.8624]
Mean: 0.8074  Std: 0.0508


In [18]:
print(f"Leaky CV mean ROC-AUC   : {cv_scores_leaked.mean():.4f}")
print(f"Correct CV mean ROC-AUC : {cv_scores_correct.mean():.4f}")
print(f"Difference              : {cv_scores_leaked.mean() - cv_scores_correct.mean():+.4f}")

Leaky CV mean ROC-AUC   : 0.8075
Correct CV mean ROC-AUC : 0.8074
Difference              : +0.0001


### Bonus Discussion

The difference between the two mean ROC-AUC scores above is likely **small** on this particular dataset — `StandardScaler` only shifts and rescales each feature using the mean and standard deviation, and with 600 reasonably well-behaved numeric values split across 5 folds, each fold's training partition already has a distribution very close to the full dataset's. So the "leaked" statistics barely differ from the "correct" per-fold statistics, and the resulting scaled values (and therefore the model's learned coefficients and scores) end up nearly identical.

**The risk is still real in general**, for reasons independent of how small the gap happens to be here:
- With a **smaller dataset**, **more skewed features**, or **more outliers**, each fold's training partition can have a noticeably different mean/std than the full dataset, making the leaked scaler's statistics diverge more from what each fold "should" have seen — inflating CV scores more substantially.
- The leak is much more dangerous for preprocessing steps that do more than shift/rescale — e.g., **imputation using group statistics, feature selection based on correlation with the target, or target encoding of categorical variables** — where fitting on the full dataset can leak actual *target information* into every fold, not just distributional shape.
- Cross-validation exists specifically to give an honest estimate of how the model will perform on data it hasn't seen. Fitting any transformation on the full dataset before CV silently breaks that guarantee — the fact that it happens to produce a similar number *this time* is a property of this dataset and this transformation, not a property of the leak being harmless. Wrapping every learned transformation inside the `Pipeline` passed to `cross_val_score` is the general, dataset-independent fix.